# Step 8 – Refined model: grade 1 versus grade 3 (Section 5.7, Table 15)

Leakage-free CV of 2-Stage GeS + HWNB and of the label-free proliferation score on grade 1 versus grade 3, and the final model applied to the cohort's grade 2 tumours. Run once per cohort.

> Refined genomic-grade model: 2-Stage GeS + HWNB trained on grade 1 vs grade 3 only (leakage-free 10 x 5 CV), compared with a one-feature proliferation score, then applied to grade 2 tumours that never enter training.

*Part of the code accompanying* **Explainable, Leakage-Free Gene Selection for Breast Cancer Histologic Grade** (Lamba, Munjal, Gigras). Run the notebooks from the folder that holds them, with the exported CSV files in `./data`.

**Parameters** – change and re-run for each cohort.

In [ ]:
cohort = "GSE7390"   # one of: GSE7390, UPP, MAINZ, UNT

In [ ]:
import os, sys, time, pickle, numpy as np, pandas as pd
from scipy.stats import spearmanr
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, f1_score, matthews_corrcoef, balanced_accuracy_score
from sklearn.linear_model import LogisticRegression
%run pipeline.ipynb

CG = list(pd.read_csv("./common_genes.csv").iloc[:, 0])
BUDGET = float(os.environ.get("BUDGET", 1e9))
FD = f"./folds_bin_{cohort}"; os.makedirs(FD, exist_ok=True)
X, y, P = load(cohort, genes=CG)
Xv, genes = X.values, np.array(X.columns)
i13 = np.where(y != 1)[0]; yb = (y[i13] == 2).astype(int)
aurka = list(genes).index("AURKA")
CLS = ["HWNB", "NB", "CWNB", "RF", "SVM"]

In [ ]:
def bmetrics(yt, p, score=None):
    d = {"f1": f1_score(yt, p, average="macro"), "mcc": matthews_corrcoef(yt, p), "bal": balanced_accuracy_score(yt, p),
         "sens": ((p == 1) & (yt == 1)).sum() / max((yt == 1).sum(), 1), "spec": ((p == 0) & (yt == 0)).sum() / max((yt == 0).sum(), 1)}
    if score is not None and len(np.unique(yt)) == 2:
        d["auc"] = roc_auc_score(yt, score)
    return d

In [ ]:
def prolif_score(Xtr, Xte):
    r = pd.DataFrame(Xtr).rank().corrwith(pd.Series(Xtr[:, aurka]).rank()).values
    mod = np.where(r >= 0.5)[0]
    mu, sd = Xtr[:, mod].mean(0), Xtr[:, mod].std(0) + 1e-9
    return ((Xtr[:, mod] - mu) / sd).mean(1), ((Xte[:, mod] - mu) / sd).mean(1), len(mod)

In [ ]:
t0 = time.time()
for rep in range(10):
    skf = StratifiedKFold(5, shuffle=True, random_state=rep)
    for k, (tr, te) in enumerate(skf.split(i13, yb)):
        fp = f"{FD}/r{rep}_k{k}.pkl"
        if os.path.exists(fp): continue
        if time.time() - t0 > BUDGET: print("BUDGET STOP"); sys.exit(0)
        Xtr, Xte, ytr, yte = Xv[i13[tr]], Xv[i13[te]], yb[tr], yb[te]
        r = ges_fit(Xtr, ytr, C=2, seed=rep * 10 + k)
        Dte = apply_cuts(Xte[:, r["genes"]], r["cuts"])
        rec = {"rep": rep, "k": k, "te": i13[te], "genes": genes[r["genes"]], "n_cfs": r["n_cfs"], "n_cons": r["n_cons"], "res": {}}
        for c in CLS:
            m = build(c).fit(r["Dtr"], r["ytr"], r["nb"], 2)
            p = m.predict(Dte)
            sc = m.predict_proba(Dte)[:, 1] if hasattr(m, "predict_proba") else None
            rec["res"][c] = bmetrics(yte, p, sc)
            if c == "HWNB": rec["p_hwnb"] = sc
        s_tr, s_te, nmod = prolif_score(Xtr, Xte)
        lr = LogisticRegression(class_weight="balanced").fit(s_tr[:, None], ytr)
        rec["res"]["Prolif"] = bmetrics(yte, lr.predict(s_te[:, None]), s_te); rec["nmod"] = nmod
        pickle.dump(rec, open(fp, "wb"))
        print(cohort, rep, k, r["n_cfs"], r["n_cons"], "HWNB AUC %.3f F1 %.3f | prolif AUC %.3f" %
              (rec["res"]["HWNB"].get("auc", np.nan), rec["res"]["HWNB"]["f1"], rec["res"]["Prolif"]["auc"]), "%.0fs" % (time.time() - t0), flush=True)

### Final model on all grade 1 and grade 3 tumours, applied to the (unseen) grade 2 tumours

In [ ]:
# ---- final model on all grade 1 and grade 3 tumours, applied to the (unseen) grade 2 tumours
if not os.path.exists(f"{FD}/final.pkl"):
    r = ges_fit(Xv[i13], yb, C=2, seed=99)
    m = build("HWNB").fit(r["Dtr"], r["ytr"], r["nb"], 2)
    i2 = np.where(y == 1)[0]
    pG3 = m.predict_proba(apply_cuts(Xv[i2][:, r["genes"]], r["cuts"]))[:, 1]
    s13, s2, _ = prolif_score(Xv[i13], Xv[i2])
    pickle.dump({"genes": genes[r["genes"]], "i2": i2, "pG3": pG3, "prolif2": s2, "n_cfs": r["n_cfs"], "n_cons": r["n_cons"]}, open(f"{FD}/final.pkl", "wb"))
    print("FINAL", cohort, r["n_cfs"], r["n_cons"], list(genes[r["genes"]]), "G2 G3-like share %.2f" % (pG3 > 0.5).mean())
print("DONE", cohort)